In [1]:
import os
from google.colab import userdata

# 1. Fetch the secret key you saved
hf_token = userdata.get('HUGGINGFACEHUB_API_TOKEN')

# 2. Save it as an environment variable for LangChain to find automatically
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

In [2]:
!pip install -q youtube-transcript-api

In [ ]:
!pip install langchain-community

In [ ]:
!pip install langchain-huggingface[full]

In [ ]:
!pip install faiss-cpu

In [ ]:
!pip install tiktoken

In [ ]:
!pip install python-dotenv

In [8]:
!pip install -q langchain-text-splitters

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

# Step 1: Indexing

## (a) Document Ingestion

In [10]:
link = "7ARBJQn6QkM"

In [ ]:
video_id= link  # Only ID not full url
try:
  # 1. Create an instance of the API
  api = YouTubeTranscriptApi()

  # Fetch the transcript list
  transcript_list = api.fetch(video_id, languages=['en', 'hi'])

  #Flatten it to plain text
  transcript = " ".join(chunk.text for chunk in transcript_list)
  print(transcript)
except TranscriptsDisabled:
  print("No captions available for this video")

In [12]:
#transcript_list

## (b) Text Splitting

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

In [ ]:
chunks[5]

## (c) Embedding Generation

In [ ]:
#Initialize HuggingFace Embeddings
embeddings = HuggingFaceEmbeddings()

## (d) Storing in Vector Store

In [17]:
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['6d904f40-8564-45ab-9fd6-adae4f19a242'])

# Step 2: Retrieval

In [20]:
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [ ]:
retriever

In [ ]:
retriever.invoke("What is cosmos?")

#Step 3: Augmentation

In [23]:
llm = HuggingFaceEndpoint(
    repo_id='Qwen/Qwen2.5-7B-Instruct',
    task="text-generation",
    max_new_tokens=20000,
    temperature=0.2
)
model = ChatHuggingFace(llm=llm)

In [24]:
prompt = PromptTemplate(
    template="""You are helpful assisstant.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say I don't know

    {context}
    Question:{question}

    """,
    input_variables = ['context','question']
)

In [25]:
question = "Is the topic of Omiverse discussed in this video? If yes, what was discussed?"
retrieved_docs = retriever.invoke(question)

In [26]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [27]:
#context_text

In [28]:
final_prompt = prompt.invoke({"context":context_text, "question":question})

In [29]:
#final_prompt

# Step 4: Generation

In [ ]:
answer = model.invoke(final_prompt)

In [ ]:
print(answer.content)

# Building Chain ( Another but Simple way)

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text= "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke("What is cosmos")

In [ ]:
parser =StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | model | parser

In [ ]:
main_chain.invoke("can you summarize the video in 500 words")